In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import precision_score, recall_score, f1_score

df = pd.read_csv('final_script_csvs/dataset_1_cleaned.csv', dtype=str)
df

,Unnamed: 0,text,Label,Cat_label
0,0,Next was an engaging panel on engaging citizen...,1,Expert
1,1,I would add education & psychological health t...,1,Expert
2,2,🚀 Stanford University embraces the AI boom! 🧠 ...,1,Expert
3,3,This new piece from @nitasha.bsky.social is so...,1,Expert
4,4,I think it’s a waste of time discussing whethe...,1,Expert
...,...,...,...,...
42577,42577,@afrocaucasian @forrrestjr @kirawontmiss some ...,1,Educator
42578,42578,"btw, there are various pieces of news how gpt ...",0,0
42579,42579,"rhett “mankind,” a digital artist based in aus...",0,0
42580,42580,my professor is saying she has no issue with u...,1,Educator


In [11]:
x = df['text']
binary_y = df['Label']
category_y = df['Cat_label']

In [6]:
df['Label'].value_counts()

Label
0    26607
1    15975
Name: count, dtype: int64

# Basic Models (Binary Classification)

In [7]:
# Separates the data
x_bin_train, x_bin_test, y_bin_train, y_bin_test = train_test_split(x, binary_y, test_size=0.2, shuffle=True)

In [8]:
# Uses TF-IDF transformation
tfidf = TfidfVectorizer()
bin_train_vals = tfidf.fit_transform(x_bin_train.to_numpy(dtype=str))
bin_test_vals = tfidf.transform(x_bin_test.to_numpy(dtype=str))

In [9]:
# Naive Bayes (binary class)
nb_bin = MultinomialNB()
nb_bin.fit(bin_train_vals, y_bin_train.to_numpy(dtype=int))
nb_bin_preds = nb_bin.predict(bin_test_vals)

print('Recall:', recall_score(y_bin_test.to_numpy(dtype=int), nb_bin_preds), 'Precision:', precision_score(y_bin_test.to_numpy(dtype=int), nb_bin_preds))
print('F1 Score:', f1_score(y_bin_test.to_numpy(dtype=int), nb_bin_preds))

Recall: 0.4316546762589928 Precision: 0.9857142857142858
F1 Score: 0.6003915597128562


In [10]:
# SVM (binary class)
svm_bin = LinearSVC()
svm_bin.fit(bin_train_vals, y_bin_train.to_numpy(dtype=int))
svm_bin_preds = svm_bin.predict(bin_test_vals)

print('Recall:', recall_score(y_bin_test.to_numpy(dtype=int), svm_bin_preds), 'Precision:', precision_score(y_bin_test.to_numpy(dtype=int), svm_bin_preds))
print('F1 Score:', f1_score(y_bin_test.to_numpy(dtype=int), svm_bin_preds))

Recall: 0.9565217391304348 Precision: 0.9858156028368794
F1 Score: 0.9709477694872202


# Basic Models (Categorical Classification) **OUTDATED NOW**

In [12]:
# Separates the data
x_cat_train, x_cat_test, y_cat_train, y_cat_test = train_test_split(x, category_y, test_size=0.2, shuffle=True)

In [13]:
# Uses TF-IDF transformation
tfidf = TfidfVectorizer()
cat_train_vals = tfidf.fit_transform(x_cat_train.to_numpy(dtype=str))
cat_test_vals = tfidf.transform(x_cat_test.to_numpy(dtype=str))

In [14]:
# Naive Bayes (categorical class)
nb_cat = MultinomialNB()
nb_cat.fit(cat_train_vals, y_cat_train.to_numpy(dtype=str))
nb_cat_preds = nb_cat.predict(cat_test_vals)

print('Recall:', recall_score(y_cat_test.to_numpy(dtype=str), nb_cat_preds, average='micro'), 'Precision:', precision_score(y_cat_test.to_numpy(dtype=str), nb_cat_preds, average='micro'))
print('F1 Score:', f1_score(y_cat_test.to_numpy(dtype=str), nb_cat_preds, average='micro'))

Recall: 0.7520253610426206 Precision: 0.7520253610426206
F1 Score: 0.7520253610426206


In [15]:
# SVM (categorical class)
svm_cat = LinearSVC()
svm_cat.fit(cat_train_vals, y_cat_train.to_numpy(dtype=str))
svm_cat_preds = svm_cat.predict(cat_test_vals)

print('Recall:', recall_score(y_cat_test.to_numpy(dtype=str), svm_cat_preds, average='micro'), 'Precision:', precision_score(y_cat_test.to_numpy(dtype=str), svm_cat_preds, average='micro'))
print('F1 Score:', f1_score(y_cat_test.to_numpy(dtype=str), svm_cat_preds, average='micro'))

Recall: 0.9794528589879066 Precision: 0.9794528589879066
F1 Score: 0.9794528589879066


# BERTweet Model (Binary Classification)

In [43]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset

In [46]:
model_name = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bertweet = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [61]:
train_df = pd.DataFrame({'text': x_bin_train,
                         'label': y_bin_train.astype(int)})
test_df = pd.DataFrame({'text': x_bin_test,
                         'label': y_bin_test.astype(int)})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [62]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/188020 [00:00<?, ? examples/s]

Map:   0%|          | 0/47005 [00:00<?, ? examples/s]

In [63]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=100,
)

In [64]:
trainer = Trainer(
    model=bertweet,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [65]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 